In [1]:
from collections.abc import Sequence
from dataclasses import dataclass
from typing import Any, Literal
from itertools import product

import pandas as pd
import numpy as np
from jaxtyping import Float
from muutils.collect_warnings import CollateWarnings
from sklearn.base import TransformerMixin
from sklearn.manifold import TSNE, Isomap
from umap import UMAP
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    SpectralClustering,
    DBSCAN,
    OPTICS,
)

# Import your existing types and functions
from spd.analysis.grouping import CoactivationResultsGroup
from spd.analysis.embedding import (
    NDArray,
    get_comp_dist_mat,
    get_embedding_model,
    get_clustering_model,
    ReduceMethod,
    ClusteringMethod,
)

from pathlib import Path

import pandas as pd
import torch
import matplotlib.pyplot as plt

from spd.analysis.embedding import get_comp_dist_mat, plot_embedding_result, sweep_embedding_param
from spd.analysis.grouping import (
    CoactivationResults,
	CoactivationResultsGroup,
    get_coactivations,
    coactivation_hierarchical_clustering,
)
from spd.data_utils import SparseFeatureDataset
from spd.experiments.resid_mlp.resid_mlp_dataset import ResidualMLPDataset
from spd.utils import get_device
from spd.analysis.grouping import print_coac_info
from spd.analysis.embed_vis import coactivation_analysis

from muutils.dbg import dbg_auto, dbg_tensor
from js_embedding_vis import fetch_jev, write_inlined_config
from spd.analysis.embed_vis import AnalysisConfig


/home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [3]:
DEVICE = get_device()
torch.set_grad_enabled(False)
print(f"Using device: {DEVICE = }")




Using device: DEVICE = 'cuda'


In [4]:

coactivations_tms: CoactivationResults = get_coactivations(
    model_path=Path("../data/tms-decomp/model_40000.pth"),
    dataset_cls=SparseFeatureDataset,
    dataset_kwargs=dict(
        value_range=(0.0, 1.0),
        synced_inputs=None,
    ),
    coactivations_kwargs=dict(
        module_groups=[["linear1", "linear2"]],
    ),
    device=DEVICE,
)


In [5]:

# Full analysis



df: pd.DataFrame
with CollateWarnings(fmt="({count}x) {filename}:{lineno}\n  {category}: {message}"):
    df, metadata = coactivation_analysis(
        group=coactivations_tms["group_0"],
        config = AnalysisConfig(
            embedding_configs={
                "umap": {"n_neighbors": [8, 16, 32]},
                # "isomap": {"n_neighbors": [8, 16, 32]},
            },
            clustering_configs={
                "kmeans": {"n_clusters": [5, 10]},
                # "agglomerative": {"n_clusters": [5, 10]},
            },
            n_components=3,
        ),
    )

# Print structure
print(metadata.describe())

# Access specific columns
umap_coords = df[["embed.umap.n_neighbors-16.ax.0", "embed.umap.n_neighbors-16.ax.1"]]
hclust_labels = df["feat.class.hclust"]
kmeans_on_umap = df["feat.class.umap.n_neighbors-16.kmeans.n_clusters-5"]

Starting coactivation analysis...
Computing hierarchical clustering...
Working with 78 alive features out of 400 total
Computing umap embeddings...


[ /home/miv/projects/MATS/apd/spd/analysis/embedding.py:165 ] jac: μ=0.00 σ=0.03 x̃=0.00 R=[0.00,1.00] ℙ˪=|█▄▃▂▂▁▃| shape=(400,400) dtype=torch.float32 device=cpu ∇✗
[ /home/miv/projects/MATS/apd/spd/analysis/embedding.py:172 ] dist: μ=1.00 σ=0.02 x̃=1.00 R=[0.50,1.00] ℙ˪=|▃▂▃▃▃▅█| shape=(400,400) dtype=torch.float32 device=cpu ∇✗
(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132 FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/umap/umap_.py:1865 UserWarning: using precomputed metric; inverse_transform will be unavailable
(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/umap/umap_.py:1952 UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132 FutureWarnin

Computing clustering on embeddings...
Analysis complete! DataFrame shape: (400, 19)
Column groups: ['feat(3)', 'embed(9)', 'feat.class(7)']
Analysis DataFrame with 400 features (78 alive)

Column Groups:
  feat: 3 columns
    ['feat.alive', 'feat.activation_freq', 'feat.module']
  embed: 9 columns
    ['embed.umap.n_neighbors-8.ax.0', 'embed.umap.n_neighbors-8.ax.1', 'embed.umap.n_neighbors-8.ax.2', 'embed.umap.n_neighbors-16.ax.0', 'embed.umap.n_neighbors-16.ax.1', 'embed.umap.n_neighbors-16.ax.2', 'embed.umap.n_neighbors-32.ax.0', 'embed.umap.n_neighbors-32.ax.1', 'embed.umap.n_neighbors-32.ax.2']
  feat.class: 7 columns
    ['feat.class.hclust', 'feat.class.umap.n_neighbors-8.kmeans.n_clusters-5', 'feat.class.umap.n_neighbors-8.kmeans.n_clusters-10', 'feat.class.umap.n_neighbors-16.kmeans.n_clusters-5', 'feat.class.umap.n_neighbors-16.kmeans.n_clusters-10', 'feat.class.umap.n_neighbors-32.kmeans.n_clusters-5', 'feat.class.umap.n_neighbors-32.kmeans.n_clusters-10']

Embedding methods

(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132 FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/umap/umap_.py:1865 UserWarning: using precomputed metric; inverse_transform will be unavailable
(1x) /home/miv/projects/MATS/apd/.venv/lib/python3.11/site-packages/umap/umap_.py:1952 UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.


In [6]:
df

,feat.alive,feat.activation_freq,feat.module,feat.class.hclust,embed.umap.n_neighbors-8.ax.0,embed.umap.n_neighbors-8.ax.1,embed.umap.n_neighbors-8.ax.2,embed.umap.n_neighbors-16.ax.0,embed.umap.n_neighbors-16.ax.1,embed.umap.n_neighbors-16.ax.2,embed.umap.n_neighbors-32.ax.0,embed.umap.n_neighbors-32.ax.1,embed.umap.n_neighbors-32.ax.2,feat.class.umap.n_neighbors-8.kmeans.n_clusters-5,feat.class.umap.n_neighbors-8.kmeans.n_clusters-10,feat.class.umap.n_neighbors-16.kmeans.n_clusters-5,feat.class.umap.n_neighbors-16.kmeans.n_clusters-10,feat.class.umap.n_neighbors-32.kmeans.n_clusters-5,feat.class.umap.n_neighbors-32.kmeans.n_clusters-10
0,False,0.000,linear1,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
1,True,0.046,linear1,32,6.175158,2.940476,1.446329,9.059335,7.338993,7.187576,10.668759,6.136651,4.543019,4,9,2,7,1,5
2,False,0.000,linear1,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
3,False,0.000,linear1,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
4,False,0.000,linear1,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,False,0.000,linear2,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
396,True,0.067,linear2,38,6.377641,1.296262,0.890909,8.629358,6.919984,8.316503,10.891128,6.563913,3.120740,0,0,0,8,4,7
397,True,0.073,linear2,18,5.122942,2.821100,2.893649,10.395280,9.021839,6.888651,11.056912,4.539354,4.414777,2,5,1,6,3,3
398,False,0.000,linear2,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1


In [7]:
# only rows where "feat.alive" col is True
df_alive_only = df[df["feat.alive"] == True]

In [8]:
df_alive_only

,feat.alive,feat.activation_freq,feat.module,feat.class.hclust,embed.umap.n_neighbors-8.ax.0,embed.umap.n_neighbors-8.ax.1,embed.umap.n_neighbors-8.ax.2,embed.umap.n_neighbors-16.ax.0,embed.umap.n_neighbors-16.ax.1,embed.umap.n_neighbors-16.ax.2,embed.umap.n_neighbors-32.ax.0,embed.umap.n_neighbors-32.ax.1,embed.umap.n_neighbors-32.ax.2,feat.class.umap.n_neighbors-8.kmeans.n_clusters-5,feat.class.umap.n_neighbors-8.kmeans.n_clusters-10,feat.class.umap.n_neighbors-16.kmeans.n_clusters-5,feat.class.umap.n_neighbors-16.kmeans.n_clusters-10,feat.class.umap.n_neighbors-32.kmeans.n_clusters-5,feat.class.umap.n_neighbors-32.kmeans.n_clusters-10
1,True,0.046,linear1,32,6.175158,2.940476,1.446329,9.059335,7.338993,7.187576,10.668759,6.136651,4.543019,4,9,2,7,1,5
6,True,0.058,linear1,7,4.503358,2.921517,1.336271,9.550128,7.303560,9.220596,9.986023,6.028708,3.144830,1,1,0,4,4,7
7,True,0.057,linear1,35,4.424162,3.472260,2.094699,10.851697,8.160412,7.681435,10.005728,6.876104,3.490126,2,2,1,1,4,7
11,True,0.062,linear1,14,5.973648,4.114192,0.973705,10.463694,6.749833,7.688059,10.446970,4.797593,4.441333,4,8,3,2,3,3
24,True,0.048,linear1,10,4.938365,2.812454,2.106614,9.128946,7.767635,8.654976,9.372944,5.306060,4.947752,2,2,0,0,2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,True,0.055,linear2,4,5.215464,1.814071,0.493670,9.413275,8.758367,9.005740,11.594161,5.638032,3.536185,1,6,4,3,3,5
390,True,0.096,linear2,27,5.856762,3.508575,0.781019,11.274026,7.999260,8.222632,11.003285,4.992959,4.581173,4,3,1,5,3,3
393,True,0.072,linear2,2,5.507648,2.897563,0.559260,10.503117,6.431786,8.108602,11.122368,6.215393,4.935499,1,3,3,5,1,5
396,True,0.067,linear2,38,6.377641,1.296262,0.890909,8.629358,6.919984,8.316503,10.891128,6.563913,3.120740,0,0,0,8,4,7


In [9]:
# nan stats abt each column of df_alive_only
nan_stats = df_alive_only.isna().sum()
nan_stats[nan_stats > 0]

Series([], dtype: int64)

In [10]:
# convert each column starting with "feat.class." int to string
for col in df_alive_only.columns:
	if col.startswith("feat.class."):
		df_alive_only[col] = df_alive_only[col].astype(str)

/tmp/ipykernel_779846/676592811.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_alive_only[col] = df_alive_only[col].astype(str)
/tmp/ipykernel_779846/676592811.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_alive_only[col] = df_alive_only[col].astype(str)
/tmp/ipykernel_779846/676592811.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.

In [ ]:
df_jsonl = df_alive_only.to_dict(orient="records")

# df_alive_only.to_json(
# 	Path("temp.jsonl"),
# 	orient="records",
# 	lines=True,
# )

write_inlined_config(
	cfg=dict(
		dataFile=None,
		data=df_jsonl,
		numericalPrefix="embed.umap.n_neighbors-32.ax.",
		defaultColorColumn="feat.class.hclust",
		defaultSelectionColumn="feat.class.hclust",
		hoverColumns=[
			"feat.class.hclust",
			"feat.activation_freq",
		],
	),
	out_path=Path("../display/embed_vis.html")
);
